# 🥈 Silver Price Analysis (2016-2026)
### Exploratory Data Analysis & Market Insights

**Author:** Areeba Bushra  
**Data Source:** Yahoo Finance — Silver Futures (SI=F)  
**Period:** 2016 – 2026 (10 Years)

---

## 📌 What This Notebook Covers
- **Data Collection** — 10 years of silver futures data via yfinance
- **Exploratory Data Analysis** — distributions, trends, volatility
- **Technical Indicators** — Moving Averages, Bollinger Bands, RSI, MACD
- **Seasonality Analysis** — monthly & day-of-week patterns
- **Market Insights** — real-world drivers of silver price movements

---

## 1. 📦 Imports & Configuration

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E76F51',
    'accent'   : '#2A9D8F',
    'danger'   : '#E63946',
    'purple'   : '#7B2D8B',
    'silver'   : '#A8A9AD',
    'gold'     : '#F4A261',
}

print('✅ Libraries loaded successfully!')


✅ Libraries loaded successfully!


---
## 2. 📥 Data Collection

Fetching 10 years of Silver Futures (SI=F) data directly from Yahoo Finance using `yfinance`.

In [4]:

# Download
raw = yf.download('SI=F',  progress=False)
raw.reset_index(inplace=True)

# Flatten multi-level columns if present
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = [col[0] for col in raw.columns]

print(f'✅ Data downloaded: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
print(f'📅 Date range: {raw["Date"].min().date()} → {raw["Date"].max().date()}')
raw.head()

✅ Data downloaded: 22 rows × 6 columns
📅 Date range: 2026-03-19 → 2026-04-19


,Date,Close,High,Low,Open,Volume
0,2026-03-19,70.902000,76.105003,67.000000,75.705002,138
1,2026-03-20,69.360001,71.925003,69.360001,71.919998,45
2,2026-03-23,69.049004,69.510002,61.090000,64.739998,271
3,2026-03-24,69.274002,71.184998,68.550003,69.089996,122
4,2026-03-25,72.361000,73.169998,71.224998,71.389999,211


---
## 3. 🔧 Data Preprocessing & Feature Engineering

In [6]:
df = raw.copy()

# ── Date features ──────────────────────────────────────────────────────────
df['Date']        = pd.to_datetime(df['Date'])
df                = df.sort_values('Date').reset_index(drop=True)
df['Year']        = df['Date'].dt.year
df['Month']       = df['Date'].dt.month
df['Month_Name']  = df['Date'].dt.month_name()
df['Day_of_Week'] = df['Date'].dt.dayofweek
df['Day_Name']    = df['Date'].dt.day_name()
df['Quarter']     = df['Date'].dt.quarter

# ── Price features ─────────────────────────────────────────────────────────
df['Daily_Range']      = df['High'] - df['Low']
df['Daily_Return']     = df['Close'].pct_change() * 100
df['Cumulative_Return']= ((1 + df['Daily_Return'] / 100).cumprod() - 1) * 100

# ── Missing values ─────────────────────────────────────────────────────────
print('🔍 Missing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().any() else '   None ✅')
print(f'\n📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

🔍 Missing Values:
Daily_Return         1
Cumulative_Return    1
dtype: int64

📊 Dataset shape: 2,512 rows × 15 columns


,Date,Close,High,Low,Open,Volume,Year,Month,Month_Name,Day_of_Week,Day_Name,Quarter,Daily_Range,Daily_Return,Cumulative_Return
0,2016-04-20,17.131001,17.131001,16.940001,16.965000,2,2016,4,April,2,Wednesday,2,0.191,NaN,NaN
1,2016-04-21,17.086000,17.086000,17.086000,17.086000,0,2016,4,April,3,Thursday,2,0.000,-0.262682,-0.262682
2,2016-04-22,16.896000,16.896000,16.896000,16.896000,7,2016,4,April,4,Friday,2,0.000,-1.112025,-1.371786
3,2016-04-25,17.004999,17.004999,17.004999,17.004999,1,2016,4,April,0,Monday,2,0.000,0.645119,-0.735517
4,2016-04-26,17.106001,17.106001,17.106001,17.106001,2,2016,4,April,1,Tuesday,2,0.000,0.593953,-0.145932


---
## 4. 📊 Statistical Summary & Key Insights

In [7]:
# ── Core stats ────────────────────────────────────────────────────────────
stats = df[['Open','High','Low','Close','Volume','Daily_Return','Daily_Range']].describe().round(3)
stats.loc['range'] = stats.loc['max'] - stats.loc['min']
stats.loc['iqr']   = stats.loc['75%'] - stats.loc['25%']
print('📊 Statistical Summary:')
print(stats)

📊 Statistical Summary:
           Open      High       Low     Close      Volume  Daily_Return  \
count  2512.000  2512.000  2512.000  2512.000    2512.000      2511.000   
mean     24.513    24.730    24.277    24.493    1763.985         0.085   
std      12.926    13.281    12.501    12.855   10067.905         2.118   
min      12.070    12.205    11.735    11.735       0.000       -31.347   
25%      16.984    17.060    16.925    16.991      12.000        -0.846   
50%      22.172    22.332    22.023    22.175      52.500         0.093   
75%      26.177    26.331    25.951    26.111     187.250         0.974   
max     116.890   121.300   112.735   115.080  131415.000        14.025   
range   104.820   109.095   101.000   103.345  131415.000        45.372   
iqr       9.193     9.271     9.026     9.120     175.250         1.820   

       Daily_Range  
count     2512.000  
mean         0.453  
std          1.289  
min          0.000  
25%          0.055  
50%          0.205  
75% 

In [ ]:
# ── Key insights printout ─────────────────────────────────────────────────
max_row   = df.loc[df['Close'].idxmax()]
min_row   = df.loc[df['Close'].idxmin()]
first_p   = df.iloc[0]['Close']
last_p    = df.iloc[-1]['Close']
total_ret = ((last_p - first_p) / first_p) * 100

print('=' * 60)
print('💡 KEY INSIGHTS — SILVER PRICE (2016–2026)')
print('=' * 60)
print(f'  🔺 All-Time High  : ${df["High"].max():.2f}  on {max_row["Date"].strftime("%Y-%m-%d")}')
print(f'  🔻 All-Time Low   : ${df["Low"].min():.2f}  on {min_row["Date"].strftime("%Y-%m-%d")}')
print(f'  📌 Current Price  : ${last_p:.2f}')
print(f'  📈 10-Year Return : {total_ret:.2f}%  (${first_p:.2f} → ${last_p:.2f})')
print(f'  📅 Trading Days   : {len(df):,}')
print()
print(f'  📊 DAILY RETURNS')
print(f'     Best Day    : +{df["Daily_Return"].max():.2f}%')
print(f'     Worst Day   : {df["Daily_Return"].min():.2f}%')
print(f'     Average     :  {df["Daily_Return"].mean():.4f}%')
print(f'     Volatility  :  {df["Daily_Return"].std():.4f}%')
print(f'     Win Rate    :  {(df["Daily_Return"] > 0).mean()*100:.1f}% positive days')

---
## 5. 📈 Price Trend & Moving Averages

In [ ]:
# ── Calculate moving averages ──────────────────────────────────────────────
df['MA_20']  = df['Close'].rolling(20).mean()
df['MA_50']  = df['Close'].rolling(50).mean()
df['MA_200'] = df['Close'].rolling(200).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

# Price + MAs
ax1 = axes[0]
ax1.plot(df['Date'], df['Close'],  color=COLORS['silver'],    lw=1,   alpha=0.8, label='Close Price')
ax1.plot(df['Date'], df['MA_20'],  color=COLORS['primary'],   lw=1.5, label='20-Day MA')
ax1.plot(df['Date'], df['MA_50'],  color=COLORS['secondary'], lw=1.5, label='50-Day MA')
ax1.plot(df['Date'], df['MA_200'], color=COLORS['danger'],    lw=2,   label='200-Day MA')
ax1.fill_between(df['Date'], df['MA_20'], df['MA_50'],
                 where=df['MA_20'] > df['MA_50'], alpha=0.15, color='green', label='Bullish Zone')
ax1.fill_between(df['Date'], df['MA_20'], df['MA_50'],
                 where=df['MA_20'] <= df['MA_50'], alpha=0.15, color='red', label='Bearish Zone')
ax1.set_title('🥈 Silver Price with Moving Averages (2016–2026)', fontsize=15, fontweight='bold')
ax1.set_ylabel('Price (USD/oz)')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# Volume
ax2 = axes[1]
colors_vol = ['green' if r > 0 else 'red' for r in df['Daily_Return'].fillna(0)]
ax2.bar(df['Date'], df['Volume'], color=colors_vol, alpha=0.5, width=1)
ax2.set_title('📦 Trading Volume', fontsize=14, fontweight='bold')
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_price_trend.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. 📉 Price & Returns Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Close price histogram
ax = axes[0, 0]
ax.hist(df['Close'], bins=60, color=COLORS['primary'], edgecolor='white', alpha=0.85)
ax.axvline(df['Close'].mean(),   color='red',   ls='--', lw=2, label=f'Mean  ${df["Close"].mean():.2f}')
ax.axvline(df['Close'].median(), color='green', ls='--', lw=2, label=f'Median ${df["Close"].median():.2f}')
ax.set_title('Closing Price Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Price (USD/oz)')
ax.set_ylabel('Frequency')
ax.legend()

# Box plot by year
ax = axes[0, 1]
df.boxplot(column='Close', by='Year', ax=ax,
           boxprops=dict(color=COLORS['primary']),
           medianprops=dict(color='red', linewidth=2))
ax.set_title('Price Distribution by Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Price (USD/oz)')
plt.suptitle('')

# Daily returns histogram
ax = axes[1, 0]
ax.hist(df['Daily_Return'].dropna(), bins=100, color=COLORS['accent'], edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=1.5)
ax.axvline(df['Daily_Return'].mean(), color='red', ls='--', lw=2,
           label=f'Mean {df["Daily_Return"].mean():.4f}%')
ax.set_title('Daily Returns Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Frequency')
ax.legend()

# Cumulative return
ax = axes[1, 1]
ax.plot(df['Date'], df['Cumulative_Return'], color=COLORS['secondary'], lw=1.5)
ax.fill_between(df['Date'], df['Cumulative_Return'], alpha=0.2, color=COLORS['secondary'])
ax.axhline(0, color='black', lw=1)
ax.set_title('Cumulative Return (2016–2026)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')

plt.tight_layout()
plt.savefig('02_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. 🌊 Volatility & Bollinger Bands

In [ ]:
# ── Bollinger Bands & rolling volatility ──────────────────────────────────
df['BB_Mid']   = df['Close'].rolling(20).mean()
df['BB_Std']   = df['Close'].rolling(20).std()
df['BB_Upper'] = df['BB_Mid'] + 2 * df['BB_Std']
df['BB_Lower'] = df['BB_Mid'] - 2 * df['BB_Std']
df['Volatility_20'] = df['Daily_Return'].rolling(20).std()
df['ATR']      = df['Daily_Range'].rolling(14).mean()

# Show last 500 trading days for clarity
recent = df.tail(500).copy()

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# Bollinger Bands
ax1 = axes[0]
ax1.plot(recent['Date'], recent['Close'],    color=COLORS['primary'],   lw=1.5, label='Close')
ax1.plot(recent['Date'], recent['BB_Upper'], color=COLORS['danger'],    lw=1,   ls='--', label='Upper Band')
ax1.plot(recent['Date'], recent['BB_Mid'],   color=COLORS['secondary'], lw=1,   ls='--', label='Middle Band')
ax1.plot(recent['Date'], recent['BB_Lower'], color=COLORS['accent'],    lw=1,   ls='--', label='Lower Band')
ax1.fill_between(recent['Date'], recent['BB_Lower'], recent['BB_Upper'], alpha=0.1, color='gray')
ax1.set_title('📊 Bollinger Bands — Last 500 Trading Days', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD/oz)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Rolling volatility
ax2 = axes[1]
ax2.plot(df['Date'], df['Volatility_20'], color=COLORS['purple'], lw=1.5)
ax2.fill_between(df['Date'], df['Volatility_20'], alpha=0.25, color=COLORS['purple'])
ax2.set_title('📉 20-Day Rolling Volatility (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Std Dev of Returns (%)')
ax2.grid(True, alpha=0.3)

# ATR
ax3 = axes[2]
ax3.plot(df['Date'], df['ATR'], color=COLORS['gold'], lw=1.5)
ax3.fill_between(df['Date'], df['ATR'], alpha=0.25, color=COLORS['gold'])
ax3.set_title('📐 Average True Range — 14 Day (ATR)', fontsize=14, fontweight='bold')
ax3.set_ylabel('ATR (USD)')
ax3.set_xlabel('Date')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('03_volatility_bollinger.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 📡 Technical Indicators — RSI & MACD

In [ ]:
# ── RSI ────────────────────────────────────────────────────────────────────
delta    = df['Close'].diff()
gain     = delta.where(delta > 0, 0).rolling(14).mean()
loss     = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))

# ── MACD ───────────────────────────────────────────────────────────────────
ema12         = df['Close'].ewm(span=12, adjust=False).mean()
ema26         = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']    = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

recent = df.tail(500).copy()

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# Price
ax1 = axes[0]
ax1.plot(recent['Date'], recent['Close'], color=COLORS['primary'], lw=1.5, label='Close Price')
ax1.set_title('🥈 Silver Close Price — Last 500 Days', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD/oz)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RSI
ax2 = axes[1]
ax2.plot(recent['Date'], recent['RSI'], color=COLORS['purple'], lw=1.5)
ax2.axhline(70, color=COLORS['danger'], ls='--', lw=1.5, label='Overbought (70)')
ax2.axhline(30, color=COLORS['accent'], ls='--', lw=1.5, label='Oversold (30)')
ax2.axhline(50, color='gray', ls='-',  lw=0.8)
ax2.fill_between(recent['Date'], 30, 70, alpha=0.08, color='gray')
ax2.set_ylim(0, 100)
ax2.set_title('💹 RSI — Relative Strength Index (14)', fontsize=14, fontweight='bold')
ax2.set_ylabel('RSI')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# MACD
ax3 = axes[2]
ax3.plot(recent['Date'], recent['MACD'],        color=COLORS['primary'],   lw=1.5, label='MACD')
ax3.plot(recent['Date'], recent['MACD_Signal'], color=COLORS['secondary'], lw=1.5, label='Signal')
hist_colors = ['green' if v > 0 else 'red' for v in recent['MACD_Hist']]
ax3.bar(recent['Date'], recent['MACD_Hist'], color=hist_colors, alpha=0.5, width=1)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_title('📊 MACD — Moving Average Convergence Divergence', fontsize=14, fontweight='bold')
ax3.set_ylabel('MACD')
ax3.set_xlabel('Date')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('04_rsi_macd.png', dpi=150, bbox_inches='tight')
plt.show()

# Current status
rsi_now  = df.iloc[-1]['RSI']
macd_now = df.iloc[-1]['MACD']
sig_now  = df.iloc[-1]['MACD_Signal']
print(f'📡 CURRENT TECHNICAL STATUS')
print(f'   RSI  : {rsi_now:.2f}  → {"Overbought 🔴" if rsi_now>70 else "Oversold 🟢" if rsi_now<30 else "Neutral ⚪"}')
print(f'   MACD : {macd_now:.4f}  → {"Bullish 🟢" if macd_now > sig_now else "Bearish 🔴"}')

---
## 9. 📅 Yearly Performance Analysis

In [ ]:
yearly = df.groupby('Year').agg(
    Open_Price  = ('Close', 'first'),
    Close_Price = ('Close', 'last'),
    Avg_Price   = ('Close', 'mean'),
    Min_Price   = ('Close', 'min'),
    Max_Price   = ('Close', 'max'),
    Volatility  = ('Close', 'std'),
    Total_Volume= ('Volume', 'sum'),
    Avg_Return  = ('Daily_Return', 'mean'),
).round(2)

yearly['Annual_Return_%'] = ((yearly['Close_Price'] - yearly['Open_Price']) / yearly['Open_Price'] * 100).round(2)
yearly['Price_Range']     = yearly['Max_Price'] - yearly['Min_Price']

print('📅 YEARLY PERFORMANCE SUMMARY')
print('=' * 80)
print(yearly.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

years   = yearly.index.tolist()
returns = yearly['Annual_Return_%'].values

# Annual returns bar
ax1 = axes[0, 0]
colors_yr = [COLORS['accent'] if r > 0 else COLORS['danger'] for r in returns]
bars = ax1.bar(years, returns, color=colors_yr, edgecolor='black', lw=0.5)
ax1.axhline(0, color='black', lw=1)
for bar, val in zip(bars, returns):
    ax1.annotate(f'{val:.1f}%',
                 xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 4 if val >= 0 else -12), textcoords='offset points',
                 ha='center', fontsize=9, fontweight='bold')
ax1.set_title('📈 Annual Return (%)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Year'); ax1.set_ylabel('Return (%)')
ax1.grid(True, alpha=0.3, axis='y')

# Avg price with min-max range
ax2 = axes[0, 1]
ax2.plot(years, yearly['Avg_Price'], marker='o', lw=2, ms=8, color=COLORS['primary'])
ax2.fill_between(years, yearly['Min_Price'], yearly['Max_Price'], alpha=0.2, color=COLORS['primary'])
ax2.set_title('💰 Avg Price by Year (with Min–Max Range)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Year'); ax2.set_ylabel('Price (USD/oz)')
ax2.grid(True, alpha=0.3)

# Volatility by year
ax3 = axes[1, 0]
ax3.bar(years, yearly['Volatility'], color=COLORS['purple'], edgecolor='black', lw=0.5)
ax3.set_title('📉 Price Volatility by Year (Std Dev)', fontsize=13, fontweight='bold')
ax3.set_xlabel('Year'); ax3.set_ylabel('Std Dev (USD)')
ax3.grid(True, alpha=0.3, axis='y')

# Volume by year
ax4 = axes[1, 1]
ax4.bar(years, yearly['Total_Volume']/1e6, color=COLORS['gold'], edgecolor='black', lw=0.5)
ax4.set_title('📦 Total Trading Volume by Year (Millions)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Year'); ax4.set_ylabel('Volume (M)')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('05_yearly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. 🗓️ Monthly Seasonality Analysis

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly = df.groupby('Month').agg(
    Avg_Return  = ('Daily_Return', 'mean'),
    Volatility  = ('Daily_Return', 'std'),
    Avg_Price   = ('Close', 'mean'),
    Avg_Volume  = ('Volume', 'mean'),
).round(4)

monthly['Win_Rate_%'] = df.groupby('Month').apply(
    lambda x: (x['Daily_Return'] > 0).mean() * 100
).round(2).values
monthly.index = month_names

print('🗓️ MONTHLY SEASONALITY')
print('=' * 70)
print(monthly.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Avg monthly return
ax1 = axes[0, 0]
colors_m = [COLORS['accent'] if r > 0 else COLORS['danger'] for r in monthly['Avg_Return']]
ax1.bar(month_names, monthly['Avg_Return'], color=colors_m, edgecolor='black', lw=0.5)
ax1.axhline(0, color='black', lw=1)
ax1.set_title('📆 Average Daily Return by Month (%)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month'); ax1.set_ylabel('Avg Return (%)')
ax1.grid(True, alpha=0.3, axis='y')

# Win rate
ax2 = axes[0, 1]
colors_wr = [COLORS['accent'] if w > 50 else COLORS['danger'] for w in monthly['Win_Rate_%']]
ax2.bar(month_names, monthly['Win_Rate_%'], color=colors_wr, edgecolor='black', lw=0.5)
ax2.axhline(50, color='black', ls='--', lw=1.5, label='50% Line')
ax2.set_title('🎯 Win Rate by Month (% Positive Days)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Month'); ax2.set_ylabel('Win Rate (%)')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')

# Box plot of returns by month
ax3 = axes[1, 0]
df.boxplot(column='Daily_Return', by='Month', ax=ax3,
           boxprops=dict(color=COLORS['primary']),
           medianprops=dict(color='red', linewidth=2))
ax3.set_title('📊 Daily Return Distribution by Month', fontsize=13, fontweight='bold')
ax3.set_xlabel('Month'); ax3.set_ylabel('Daily Return (%)')
ax3.set_xticklabels(month_names)
plt.suptitle('')

# Monthly avg price heatmap by year
ax4 = axes[1, 1]
pivot = df.pivot_table(values='Close', index='Year', columns='Month', aggfunc='mean')
pivot.columns = month_names
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4,
            cbar_kws={'label': 'Avg Price (USD/oz)'})
ax4.set_title('🗺️ Monthly Avg Price Heatmap by Year', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('06_monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. 📆 Day-of-Week Analysis

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']

dow = df.groupby('Day_Name').agg(
    Avg_Return = ('Daily_Return', 'mean'),
    Volatility = ('Daily_Return', 'std'),
    Avg_Volume = ('Volume', 'mean'),
).round(4).reindex(day_order)

dow['Win_Rate_%'] = df.groupby('Day_Name').apply(
    lambda x: (x['Daily_Return'] > 0).mean() * 100
).round(2).reindex(day_order).values

print('📆 DAY-OF-WEEK ANALYSIS')
print('=' * 60)
print(dow.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
days = ['Mon','Tue','Wed','Thu','Fri']

# Avg return by day
ax1 = axes[0]
colors_d = [COLORS['accent'] if r > 0 else COLORS['danger'] for r in dow['Avg_Return']]
ax1.bar(days, dow['Avg_Return'], color=colors_d, edgecolor='black', lw=0.5)
ax1.axhline(0, color='black', lw=1)
ax1.set_title('📊 Avg Return by Day', fontsize=13, fontweight='bold')
ax1.set_ylabel('Avg Return (%)')
ax1.grid(True, alpha=0.3, axis='y')

# Win rate by day
ax2 = axes[1]
colors_dw = [COLORS['accent'] if w > 50 else COLORS['danger'] for w in dow['Win_Rate_%']]
ax2.bar(days, dow['Win_Rate_%'], color=colors_dw, edgecolor='black', lw=0.5)
ax2.axhline(50, color='black', ls='--', lw=1.5)
ax2.set_title('🎯 Win Rate by Day', fontsize=13, fontweight='bold')
ax2.set_ylabel('Win Rate (%)')
ax2.grid(True, alpha=0.3, axis='y')

# Avg volume by day
ax3 = axes[2]
ax3.bar(days, dow['Avg_Volume'], color=COLORS['secondary'], edgecolor='black', lw=0.5)
ax3.set_title('📦 Avg Volume by Day', fontsize=13, fontweight='bold')
ax3.set_ylabel('Volume')
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('07_day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. 🔗 Correlation Analysis

In [ ]:
corr_cols = ['Open','High','Low','Close','Volume','Daily_Range','Daily_Return',
             'MA_20','MA_50','RSI','MACD']

corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('🔗 Correlation Matrix — Silver Price Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('08_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. 🌍 Market Insights — Why Silver Is Rising (2024–2026)

Based on comprehensive market research, here are the real-world drivers behind silver's price movements:

---

### 🔋 1. Solar Panel Demand — Primary Driver
- **230+ million oz** of silver consumed by the solar industry in 2024
- Solar now represents **29% of all industrial silver demand** (up from just 11% in 2014)
- New **TOPCon solar technology** requires 50% more silver per panel than older designs
- **4,000+ GW** of new solar capacity is expected to be installed globally between 2024–2030

---

### ⚡ 2. Electric Vehicle (EV) Growth
- EVs use **2–3× more silver** than traditional internal combustion engine (ICE) vehicles (25–50 grams/vehicle)
- EV silver demand grew **20% in 2025**
- EVs are expected to become the primary automotive silver consumer by 2027

---

### 🤖 3. AI & Data Center Expansion
- Silver is critical for high-efficiency electrical contacts and thermal management in servers
- AI infrastructure build-out (GPUs, cooling systems) is creating new demand streams
- Data center construction is accelerating globally, directly boosting industrial silver demand

---

### 📉 4. Persistent Supply Deficits
- 2025 marked the **5th consecutive year** of global silver supply deficit
- Deficit projected at **115–120 million oz** annually
- **70% of silver is a byproduct** of copper, zinc, and lead mining — supply cannot respond quickly to price increases
- Mine production is structurally unable to keep pace with industrial demand growth

---

### 🛡️ 5. Safe-Haven Appeal
- Economic uncertainty and geopolitical tensions are driving investment demand
- Expected interest rate cuts make non-yielding assets like silver more attractive
- A weakening US dollar is historically bullish for silver prices

---

### 📊 Price Forecast Scenarios for 2026

| Scenario | Source | Forecast |
|---|---|---|
| Conservative | Major Banks (Average) | $56/oz |
| Base Case | Technical Analysis | $72–$88/oz |
| Bullish | Analyst Consensus | $100+/oz |

> **Note:** These are market forecasts collected from public sources — not model predictions from this notebook.

---
## 14. 📋 Executive Summary

In [ ]:
print('=' * 70)
print('📋 EXECUTIVE SUMMARY — SILVER PRICE ANALYSIS (2016–2026)')
print('=' * 70)

first_p   = df.iloc[0]['Close']
last_p    = df.iloc[-1]['Close']
total_ret = ((last_p - first_p) / first_p) * 100

best_year  = yearly['Annual_Return_%'].idxmax()
worst_year = yearly['Annual_Return_%'].idxmin()
best_month  = monthly['Avg_Return'].idxmax()
worst_month = monthly['Avg_Return'].idxmin()
best_day   = dow['Avg_Return'].idxmax()
worst_day  = dow['Avg_Return'].idxmin()
rsi_now    = df.iloc[-1]['RSI']
macd_now   = df.iloc[-1]['MACD']
sig_now    = df.iloc[-1]['MACD_Signal']

print(f"""
📅 PERIOD
   Start : {df['Date'].min().strftime('%Y-%m-%d')}
   End   : {df['Date'].max().strftime('%Y-%m-%d')}
   Days  : {len(df):,} trading days

💰 PRICE
   All-Time High  : ${df['High'].max():.2f}
   All-Time Low   : ${df['Low'].min():.2f}
   Current Price  : ${last_p:.2f}
   Average Price  : ${df['Close'].mean():.2f}

📈 PERFORMANCE
   10-Year Return : {total_ret:.2f}%  (${first_p:.2f} → ${last_p:.2f})
   Best Day       : +{df['Daily_Return'].max():.2f}%
   Worst Day      :  {df['Daily_Return'].min():.2f}%
   Win Rate       : {(df['Daily_Return'] > 0).mean()*100:.1f}% positive days
   Daily Vol      : {df['Daily_Return'].std():.4f}%

📅 SEASONALITY
   Best Year   : {best_year}  ({yearly.loc[best_year, 'Annual_Return_%']:.1f}%)
   Worst Year  : {worst_year} ({yearly.loc[worst_year, 'Annual_Return_%']:.1f}%)
   Best Month  : {best_month}  (avg {monthly.loc[best_month, 'Avg_Return']:.4f}%/day)
   Worst Month : {worst_month} (avg {monthly.loc[worst_month, 'Avg_Return']:.4f}%/day)
   Best Day    : {best_day}    (avg {dow.loc[best_day, 'Avg_Return']:.4f}%/day)
   Worst Day   : {worst_day}   (avg {dow.loc[worst_day, 'Avg_Return']:.4f}%/day)

📡 CURRENT TECHNICAL SIGNALS
   RSI  ({rsi_now:.1f}) : {'Overbought 🔴' if rsi_now > 70 else 'Oversold 🟢' if rsi_now < 30 else 'Neutral ⚪'}
   MACD ({macd_now:.4f}) : {'Bullish 🟢' if macd_now > sig_now else 'Bearish 🔴'}
""")
print('=' * 70)
print('✅ Analysis Complete!')
print('=' * 70)